# Production Delivery Engineering — Hands-On

**Software Engineering · Week 01+**

Offline simulations of CI/CD workflow validation, pre-commit gating, release-note derivation, feature flags, and rollout decisions.

## 0. Setup: workflow string and validators

In [ ]:
import re, hashlib
from dataclasses import dataclass, field

WORKFLOW = '''
name: ci
permissions:
  contents: read
jobs:
  test:
    strategy:
      matrix:
        python-version: ['3.11', '3.12']
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          cache: pip
      - run: ruff check . && black --check . && mypy src
      - run: pytest --cov=src --cov-fail-under=85
      - run: bandit -r src && pip-audit
      - uses: actions/upload-artifact@v4
  deploy:
    needs: test
    environment: staging
'''
print(WORKFLOW.splitlines()[1])

## 1. Validate production controls in workflow text

In [ ]:
checks = {
    "least_privilege": r"permissions:\s*\n\s*contents: read",
    "python_matrix": r"python-version:.*3\.11.*3\.12",
    "cache": r"cache: pip",
    "coverage": r"--cov-fail-under=85",
    "security": r"bandit -r src && pip-audit",
    "artifact": r"actions/upload-artifact@v4",
    "environment": r"environment: staging",
}
result = {name: bool(re.search(pattern, WORKFLOW, re.S)) for name, pattern in checks.items()}
print(result)
assert all(result.values())

## 2. Pre-commit gate placement

In [ ]:
hooks = ["ruff", "black", "mypy", "trufflehog"]
ci_only = ["pytest-cov", "bandit", "pip-audit", "artifact-build"]
print("local fast hooks:", hooks)
print("clean-room CI checks:", ci_only)

## 3. Conventional commits become release notes

In [ ]:
commits = ["feat: add tenant flag rollout", "fix: prevent duplicate webhook", "docs: update runbook", "feat!: remove v1 export API"]
level = "patch"
notes = []
for c in commits:
    if c.startswith("feat!") or "BREAKING CHANGE" in c:
        level = "major"
    elif c.startswith("feat") and level != "major":
        level = "minor"
    if c.startswith(("feat", "fix")):
        notes.append("- " + c)
print("release:", level)
print("\n".join(notes))

## 4. Feature flag rollout evaluator

In [ ]:
@dataclass(frozen=True)
class FeatureFlag:
    key: str
    enabled: bool
    rollout_percent: int = 0
    allow_tenants: set[str] = field(default_factory=set)

def bucket(flag_key, user_id):
    digest = hashlib.sha256(f"prod:{flag_key}:{user_id}".encode()).hexdigest()
    return int(digest[:8], 16) % 100

def evaluate(flag, user_id, tenant_id):
    return flag.enabled and (tenant_id in flag.allow_tenants or bucket(flag.key, user_id) < flag.rollout_percent)

flag = FeatureFlag("llm_answer_v2", True, 30, {"acme"})
print([(u, bucket(flag.key, u), evaluate(flag, u, "beta")) for u in ["u1", "u2", "u3", "u4"]])
print("tenant override:", evaluate(flag, "anyone", "acme"))

## 5. Deployment strategy decision helper

In [ ]:
def choose_strategy(stateful, capacity_headroom, need_fast_rollback, traffic):
    if stateful:
        return "rolling plus expand/contract DB migration"
    if need_fast_rollback and capacity_headroom >= 2.0:
        return "blue/green"
    if traffic >= 1000:
        return "canary with SLO auto-abort"
    return "rolling with smoke tests"

for args in [(False, 2.0, True, 500), (False, 1.2, False, 5000), (True, 2.0, True, 200)]:
    print(args, "->", choose_strategy(*args))

## 6. Coverage and mutation signal toy model

In [ ]:
files = {"core.py": (96, 82), "api.py": (88, 51), "cli.py": (72, 20)}  # coverage %, mutation %
for name, (coverage, mutation) in files.items():
    risk = "high" if coverage >= 80 and mutation < 60 else "normal"
    print(name, "coverage", coverage, "mutation", mutation, "risk", risk)

## Exercises
1. Add a required OIDC permission check for cloud deploys.
2. Change the rollout to 5%, 25%, 50%, 100% and observe stable buckets.
3. Decide which checks should block a hotfix PR.
4. Add flag expiry metadata and fail stale flags.

## Links
- Literature note: `02 Literature Notes/Software Engineering/Production Delivery Engineering`
- Snippets: `04 Code Snippets/Software Engineering/SE Week 01+ GitHub Actions Delivery Workflow Validator`, `.../SE Week 01+ Deterministic Feature Flag Rollout Evaluator`
- MOC: `06 Maps of Content/Software Engineering Concepts`